# Game-of-24 GRPO — diagnostics

Loads `eval_rollout.jsonl` produced by `script/run_game24_one.py` and renders
**three** figures, each as a single cell:

1. **D1 · length / diversity** — mean CoT length, length split by correctness,
   and within-puzzle pairwise edit-distance of correct rollouts, all as a
   function of training step (`global_step`).
2. **Accuracy · pass@k** — pass@1, pass@4, pass@8 per eval cycle, computed
   from the JSONL alone via the unbiased combinatorial estimator
   (no extra rollouts needed; each eval cycle already has `num_generations=8`
   samples per puzzle).
3. **R_T · decoding-velocity dynamics** — for a chosen eval cycle
   (`STEP_IDX`), shows (a) one correct + one incorrect rollout's cumulative
   R_t, and (b) the global mean cumulative R_t across many rollouts of each
   class. Use this to inspect how training reshapes the trajectory.

The v_t kernel itself is the vectorized one in `src/velocity.py`.


In [ ]:
"""Setup: read eval_rollout.jsonl. No model, no GPU.

`script/run_game24_one.py --score-vt` (default) augments each rollout row
with R_T, R_per_token, and a fixed-grid cumR_resampled. All three figures
below run from those fields plus the original (numbers, correct, n_tokens,
completion, global_step) columns.
"""
import json
from pathlib import Path
from math import comb

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Point this at the output directory produced by script/run_game24_one.py
RUN_DIR  = Path("output/game24_sweep/len512/Qwen__Qwen3-0.6B")
EVAL_LOG = RUN_DIR / "eval_rollout.jsonl"
assert EVAL_LOG.exists(), f"missing {EVAL_LOG}; run script/run_game24_one.py first"

rows = [json.loads(l) for l in EVAL_LOG.read_text().splitlines() if l.strip()]
eval_df = pd.DataFrame(rows)
eval_df["key"] = eval_df["numbers"].apply(lambda x: tuple(sorted(x)))

has_vt = "R_T" in eval_df.columns and eval_df["R_T"].notna().any()
print(f"{len(eval_df)} eval rollouts across "
      f"{eval_df.global_step.nunique()} eval cycles "
      f"(global_step ∈ {sorted(eval_df.global_step.unique().tolist())})")
print(f"R_T fields present: {has_vt}")
eval_df.head(3)

In [ ]:
"""Figure 1 · D1: length / diversity / split-by-correctness over eval cycles.

All three panels group rollouts by `global_step` so each eval cycle is one
point on the x-axis. The diversity panel averages per-puzzle mean pairwise
edit-distance across puzzles (within-puzzle diversity, then per-step mean).
"""
import difflib

len_col = "n_cot_tokens" if "n_cot_tokens" in eval_df.columns else "n_tokens"

# --- panel 1: mean / p90 CoT length per step --------------------------------
agg = eval_df.groupby("global_step").agg(
    mean=(len_col, "mean"),
    p90 =(len_col, lambda s: float(np.percentile(s, 90))),
).reset_index()

# --- panel 2: length split by correctness -----------------------------------
by_corr = eval_df.groupby(["global_step", "correct"])[len_col].mean().unstack()

# --- panel 3: within-puzzle pairwise edit-dist of correct rollouts ----------
def _norm_edit(a, b):
    return 1.0 - difflib.SequenceMatcher(None, a, b, autojunk=False).ratio()

ed_rows = []
for (step, key), g in eval_df[eval_df.correct].groupby(["global_step", "key"]):
    texts = g.completion.tolist()
    if len(texts) < 2:
        continue
    pairs = [_norm_edit(texts[i], texts[j])
             for i in range(len(texts)) for j in range(i + 1, len(texts))]
    ed_rows.append({"step": step, "key": key, "mean_edit": float(np.mean(pairs))})
edf = pd.DataFrame(ed_rows)
edf_step = (edf.groupby("step")["mean_edit"].agg(["mean", "std", "count"]).reset_index()
            if len(edf) else pd.DataFrame(columns=["step", "mean", "std", "count"]))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(agg.global_step, agg["mean"], label="mean")
axes[0].plot(agg.global_step, agg["p90"],  label="p90", linestyle="--")
axes[0].set(xlabel="global_step", ylabel=f"{len_col}", title="D1 · CoT length")
axes[0].legend()

for col in by_corr.columns:
    axes[1].plot(by_corr.index, by_corr[col],
                 label="correct" if col else "incorrect")
axes[1].set(xlabel="global_step", ylabel=f"mean {len_col}",
            title="D1 · length, split by correctness")
axes[1].legend()

if len(edf_step):
    x  = edf_step["step"].values
    mu = edf_step["mean"].values
    sd = edf_step["std"].fillna(0.0).values
    axes[2].plot(x, mu, color="#264653", linewidth=2, label="mean over puzzles")
    axes[2].fill_between(x, mu - sd, mu + sd, color="#264653", alpha=0.15,
                         label="±1σ across puzzles")
    axes[2].set(xlabel="global_step",
                ylabel="within-puzzle mean pairwise edit-dist",
                title="D1 · diversity of correct CoTs (collapse → 0)")
    axes[2].legend(fontsize=8)
else:
    axes[2].text(0.5, 0.5, "not enough correct rollouts\non same puzzle",
                 ha="center", va="center"); axes[2].axis("off")

fig.tight_layout(); plt.show()


In [ ]:
"""Figure 2 · pass@1, pass@4, pass@8 per eval cycle.

Each (eval cycle, puzzle) cell already has `num_generations` rollouts (=8
by default), so we can compute the unbiased pass@k estimator
    pass@k = 1 - C(n-c, k) / C(n, k),  n = #rollouts, c = #correct
for any k ≤ n, then average over puzzles in the cycle. No extra forward
passes needed.
"""
KS = (1, 4, 8)

def pass_at_k(c: int, n: int, k: int) -> float:
    if n - c < k:
        return 1.0
    return 1.0 - comb(n - c, k) / comb(n, k)

rows = []
for (gs, key), g in eval_df.groupby(["global_step", "key"]):
    n = len(g); c = int(g.correct.sum())
    for k in KS:
        if k > n:
            continue
        rows.append({"global_step": gs, "key": key, "k": k,
                     "pass": pass_at_k(c, n, k)})
pak = (pd.DataFrame(rows)
         .groupby(["global_step", "k"])["pass"].mean()
         .unstack("k").sort_index())

fig, ax = plt.subplots(figsize=(6, 4))
for k in KS:
    if k not in pak.columns:
        continue
    ax.plot(pak.index, pak[k], marker="o", label=f"pass@{k}")
ax.set(xlabel="global_step", ylabel="pass@k", ylim=(0, 1.02),
       title="Accuracy · pass@k over eval cycles")
ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); plt.show()

print(pak.round(3))


In [ ]:
"""Figure 3 · R_T decoding-velocity dynamics at a chosen eval cycle.

Reads `R_T` (scalar) and `cumR_resampled` (fixed-grid cumulative R_t array)
directly from eval_rollout.jsonl — no model load. The training script wrote
these via src.velocity.compute_vt_batched.

Left  — one correct + one incorrect rollout's resampled cumulative R_t.
Right — mean cumulative R_t across all rollouts at this step, ±1σ band,
        split by correctness.

Tweak STEP_IDX to inspect how training reshapes the trajectory.
"""
assert has_vt, "eval_rollout.jsonl has no R_T fields — run with --score-vt enabled."

# ====== CONFIG ============================================================
STEP_IDX  = int(eval_df.global_step.max())   # which eval cycle
PAIR_SEED = 0                                # which (correct, incorrect) pair to plot
# ==========================================================================

at_step = eval_df[eval_df.global_step == STEP_IDX].copy()
at_step = at_step[at_step.cumR_resampled.notna()]   # drop degenerate rollouts
correct   = at_step[at_step.correct]
incorrect = at_step[~at_step.correct]
print(f"step={STEP_IDX}  scored correct={len(correct)}  incorrect={len(incorrect)}")

def _stack(df_class):
    if len(df_class) == 0:
        return np.empty((0, 0))
    return np.array(df_class["cumR_resampled"].tolist(), dtype=float)

R_c, R_i = _stack(correct), _stack(incorrect)
N_PTS = R_c.shape[1] if len(R_c) else (R_i.shape[1] if len(R_i) else 100)
x = np.linspace(0.0, 1.0, N_PTS)

# Individual pair sample (deterministic given PAIR_SEED).
pair_c = correct.sample(1,   random_state=PAIR_SEED).iloc[0] if len(correct)   else None
pair_i = incorrect.sample(1, random_state=PAIR_SEED).iloc[0] if len(incorrect) else None

fig, (ax_pair, ax_avg) = plt.subplots(1, 2, figsize=(14, 4.5))

# --- left: individual pair, both on the normalised CoT-position axis ------
for row, lbl, col in [(pair_c, "correct", "#2a9d8f"),
                      (pair_i, "incorrect", "#e76f51")]:
    if row is None:
        continue
    R = np.asarray(row.cumR_resampled, dtype=float)
    ax_pair.plot(x, R, color=col, linewidth=2,
                 label=f"{lbl}  (R_T={row.R_T:+.2f}, T={row.n_cot_tokens})")
ax_pair.axhline(0, color="k", linestyle=":", alpha=0.4)
ax_pair.set(xlabel="normalised CoT position (t / T)",
            ylabel="cumulative R_t",
            title=f"R_t · individual pair  (step={STEP_IDX})")
ax_pair.legend(fontsize=9); ax_pair.grid(alpha=0.3)

# --- right: global mean cumulative R_t, ±1σ -------------------------------
for R, lbl, col in [(R_c, "correct", "#2a9d8f"), (R_i, "incorrect", "#e76f51")]:
    if len(R) == 0:
        continue
    mu, sd = R.mean(0), R.std(0)
    ax_avg.plot(x, mu, color=col, linewidth=2, label=f"{lbl}  (n={len(R)})")
    ax_avg.fill_between(x, mu - sd, mu + sd, color=col, alpha=0.15)
ax_avg.axhline(0, color="k", linestyle=":", alpha=0.4)
ax_avg.set(xlabel="normalised CoT position (t / T)",
           ylabel="mean cumulative R_t",
           title=f"R_t · global average  (step={STEP_IDX})")
ax_avg.legend(fontsize=9); ax_avg.grid(alpha=0.3)

fig.tight_layout(); plt.show()